In [1]:
# Динамический импорт и инициализация библиотек и путей для работы с данными в Python <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# ''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
from pathlib import Path
import csv
import sys
import os
import pandas as pd
import json
from clickhouse_driver import Client
import gc
import numpy as np
import glob
from collections import Counter
import re
from datetime import datetime


sub_project_dir = "trading_conditions"

file_dir = os.getcwd()                                                              # Определяем путь к текущему файлу (где выполняется код)
print(f"Файл в директории:                                      {file_dir}")

project_dir                     = Path(file_dir).parent.parent                      # Переход на уровень выше (fc_to_mt5_migrations/own_platform)
print(f"Рабочая директория проекта:                             {project_dir}")

parent_dir                      = Path.cwd().parent.parent.parent                   # Переход на уровень выше (fc_to_mt5_migrations)
print(f"Рабочая директория проекта для доступа к библиотекам:   {parent_dir}")

directory_data_log_files        = os.path.join(project_dir, sub_project_dir, 'log_data_files')
print(f"[directory_data_log_files];     Путь к каталогу с лог-файлами:                          {directory_data_log_files}")

directory_data_temp_files       = os.path.join(project_dir, sub_project_dir, 'working_data_files') 
print(f"[directory_data_temp_files];    Путь к каталогу с временными файлами:                   {directory_data_temp_files}")

directory_data_original_data    = os.path.join(project_dir, 'original_data')        # Путь к каталогу с оригинальными данными
print(f"Путь к каталогу с оригинальными данными:                {directory_data_original_data}")

directory_data_set              = os.path.join(project_dir, 'data_set')             # Путь к каталогу с конфигурационными данными
print(f"Путь к каталогу с файлами настроек:                     {directory_data_set}")

directory_data_output           = os.path.join(parent_dir, 'output_data')           # Путь к каталогу с выходными данными

libraries_path = os.path.join(parent_dir, "libraries_py")                           # Формируем путь к libraries_py каталогу с библиотеками *.py
sys.path.append(libraries_path)                                                     # sys.path — это список путей, где Python ищет модули при import module_name.

if libraries_path in sys.path: print(f"✅ Каталог {libraries_path} успешно добавлен в sys.path")
else: print(f"❌ Ошибка: {libraries_path} не найден в sys.path")

# Динамически импорт необходимых функций <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
file_imports = "dynamic_import_functions.py"                                        # Библиотека для динамического импорта
file_imports_path = os.path.join(libraries_path, file_imports)
if os.path.exists(file_imports_path):
    import importlib
    importlib.invalidate_caches()                                                   # Сбрасываем кэш перед импортом
    from dynamic_import_functions import import_functions, print_import_function_info
    print(f"\n ✅ Импорт [{file_imports}] успешен.")
else:
    print(f"\n ERROR: Файл '{file_imports}' не найден по пути {file_imports_path}, импорт не выполнен.\n")

modules_to_import = {                                                           # Формируем словарь, с именами файлов и функциями в них
    "yar_sed_general_lib":
        [libraries_path,
                "pd_set_option",                        # Вывод ДФ
                "df_to_csv",                            # Сохранение ДФ в CSV 
                "CSVLoader",
                "save_data_log_work_file",
                "detect_encoding",
                "time_to_minutes",
                "load_string_list",
                "list_print",
                "move_column"],
    "sql_request_2":
        [libraries_path, 
                "pd_read_sql",
                "get_sql_tab"]
                }

imported = import_functions(modules_to_import)          # Импортируем модули из словаря modules_to_import

print_import_function_info(modules_to_import, imported) # Выводим переменные ожидаемые импортированными функциями 

Файл в директории:                                      c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\trading_conditions\ipynb_files
Рабочая директория проекта:                             c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform
Рабочая директория проекта для доступа к библиотекам:   c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations
[directory_data_log_files];     Путь к каталогу с лог-файлами:                          c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\trading_conditions\log_data_files
[directory_data_temp_files];    Путь к каталогу с временными файлами:                   c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\trading_conditions\working_data_files
Путь к каталогу с оригинальными данными:                c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\original_data
Путь к каталогу с файлами настроек:                 

In [2]:
# Преобразует текстовое расписание торговых сессий в множество (set) минут дня. <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# ''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
def get_minutes_from_session(session_str):
    """Преобразует текстовое расписание торговых сессий в множество (set) минут дня.
    Логика работы:
    1. Принимает строку расписания, поддерживая мульти-сессии через разделитель <br>.
    2. Каждое время формата ЧЧ:ММ переводит в абсолютную минуту дня (от 0 до 1439).
    3. Создает диапазон минут от начала до конца каждой сессии включительно.
    4. Возвращает уникальное множество всех "разрешенных" минут для быстрой проверки вхождения.
    
    Пример: "08:00-09:00" -> {480, 481, ..., 540}
    """
    if not isinstance(session_str, str) or not session_str.strip():
        return set()
    minutes_in_session = set()
    parts = session_str.replace('<br>', '\n').split('\n')
    for part in parts:
        if '-' in part:
            try:
                s, e = part.strip().split('-')
                start_total = int(s.split(':')[0]) * 60 + int(s.split(':')[1])
                end_total = int(e.split(':')[0]) * 60 + int(e.split(':')[1])
                for m in range(start_total, end_total + 1):
                    minutes_in_session.add(m)
            except: continue
    return minutes_in_session

# Преобразует поминутную карту плотности в список временных интервалов <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# ''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
def get_intervals(series, min_duration = 5):    
    """Преобразует поминутную карту плотности в список временных интервалов.
    Логика:
    1. Проходит по значениям серии и фиксирует моменты изменения состояния (Active <-> Gap).
    2. Если текущее состояние длится дольше min_duration, формируется строка "начало-конец".
    3. Интервалы классифицируются как 'Quotes_Presence' (наличие) или 'Quotes_Absence' (отсутствие).
    
    Пример выхода: "04.20.13:30-04.20.20:14"
    """
    active_intervals = []
    gap_intervals = []
    
    if series.empty: return "", ""

    times = series.index.tolist()
    values = series.values
    start_idx = 0
    is_active = values[0] > 0
    
    for i in range(1, len(values)):
        current_state = values[i] > 0
        if current_state != is_active:
            duration = i - start_idx                                                # Вычисляем длительность интервала в минутах
            
            if duration > min_duration:                                             # Регистрируем только если длительность больше порога
                interval_str = f"{times[start_idx]}-{times[i-1]}"
                if is_active: active_intervals.append(interval_str)
                else: gap_intervals.append(interval_str)
            
            start_idx = i
            is_active = current_state
            
    last_duration = len(values) - start_idx                                         # Проверка для последнего интервала
    if last_duration > min_duration:
        last_interval = f"{times[start_idx]}-{times[-1]}"
        if is_active: active_intervals.append(last_interval)
        else: gap_intervals.append(last_interval)
        
    return "<br>".join(active_intervals), "<br>".join(gap_intervals)



# Вспомогательная функция (или просто вынесите логику append)
def _append_outside_report(self, container, symbol, full_interval, block, sched_text, weekday):
    v_start = f"{block[0]//60:02d}:{block[0]%60:02d}"
    v_end = f"{block[-1]//60:02d}:{block[-1]%60:02d}"
    duration = block[-1] - block[0] + 1
    
    container.append({
        'Ticker': symbol,
        'Full_Interval': full_interval,
        'Violation_Period': f"{v_start}-{v_end}",
        'Duration_Min': duration,
        'Schedule_Applied': sched_text,
        'Weekday': weekday,
        'Type': 'Outside Session'
    })



# [Функция] (передаем df_sessions для получения отчёта по наличию / отсутствию котировок в разрезе расписания) <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
def analyze_status_intervals_v2(df_status, schedule_map, analysis_year, df_sessions):
    low_activity = []
    outside_activity = []

    # Создаем быстрый поиск расписания в текстовом виде для отчета
    raw_schedule = df_sessions.set_index('name_s')

    for _, row in df_status.iterrows():
        symbol = row['Ticker']
        if symbol not in schedule_map: continue


        # --- 1. ПРОВЕРКА ВНЕ СЕССИИ (Presence) ---
        if row['Quotes_Presence']:
            for interval in row['Quotes_Presence'].split('<br>'):
                start_str, end_str = interval.split('-')
                
                # Парсим полные объекты даты/времени для корректного сравнения суток
                dt_start = pd.to_datetime(f"{analysis_year}.{start_str}", format='%Y.%m.%d.%H.%M')
                dt_end = pd.to_datetime(f"{analysis_year}.{end_str}", format='%Y.%m.%d.%H.%M')
                
                csv_weekday = (dt_start.weekday() + 1) % 7
                allowed_mins = schedule_map[symbol][csv_weekday]
                sched_text = raw_schedule.loc[symbol, f"1_{csv_weekday}"]

                # Считаем базовые минуты
                start_total = dt_start.hour * 60 + dt_start.minute
                end_total = dt_end.hour * 60 + dt_end.minute
                
                # !!! НЮАНС: Обработка перехода через полночь !!!
                if dt_end.date() > dt_start.date():
                    # Если интервал захватил следующий день, увеличиваем конечную точку на 24 часа
                    days_diff = (dt_end.date() - dt_start.date()).days
                    end_total += 1440 * days_diff
                
                # Теперь range отработает корректно (например, от 1349 до 1697 минут)
                violation_mins = [m for m in range(start_total, end_total + 1) if (m % 1440) not in allowed_mins]
                
                if violation_mins:
                    current_block = [violation_mins[0]]
                    for i in range(1, len(violation_mins)):
                        if violation_mins[i] - violation_mins[i-1] > 1:
                            # При выводе используем остаток от деления на 1440, чтобы время не превышало 23:59
                            m0, mN = current_block[0] % 1440, current_block[-1] % 1440
                            v_start = f"{m0//60:02d}:{m0%60:02d}"
                            v_end = f"{mN//60:02d}:{mN%60:02d}"
                            
                            outside_activity.append({
                                'Ticker': symbol,
                                'Full_Interval': interval,
                                'Violation_Period': f"{v_start}-{v_end}",
                                'Duration_Min': len(current_block),
                                'Schedule_Applied': sched_text,
                                'Weekday': csv_weekday,
                                'Type': 'Outside Session'
                            })
                            current_block = []
                        current_block.append(violation_mins[i])
                    
                    if current_block:
                        m0, mN = current_block[0] % 1440, current_block[-1] % 1440
                        v_start = f"{m0//60:02d}:{m0%60:02d}"
                        v_end = f"{mN//60:02d}:{mN%60:02d}"
                        outside_activity.append({
                            'Ticker': symbol,
                            'Full_Interval': interval,
                            'Violation_Period': f"{v_start}-{v_end}",
                            'Duration_Min': len(current_block),
                            'Schedule_Applied': sched_text,
                            'Weekday': csv_weekday,
                            'Type': 'Outside Session'
                        })





        # --- 2. ПРОВЕРКА ДЫР ВНУТРИ СЕССИИ (Absence) ---
        if row['Quotes_Absence']:
            for interval in row['Quotes_Absence'].split('<br>'):
                start_str, end_str = interval.split('-')
                dt_start = pd.to_datetime(f"{analysis_year}.{start_str}", format='%Y.%m.%d.%H.%M')
                csv_weekday = (dt_start.weekday() + 1) % 7
                allowed_mins = schedule_map[symbol][csv_weekday]
                sched_text = raw_schedule.loc[symbol, f"1_{csv_weekday}"]
                #if symbol == 'CA60': print(sched_text)
                s_h, s_m = map(int, start_str.split('.')[-2:])
                e_h, e_m = map(int, end_str.split('.')[-2:])
                start_total, end_total = s_h * 60 + s_m, e_h * 60 + e_m

                gap_violation = [m for m in range(start_total, end_total + 1) if m in allowed_mins]
                
                if gap_violation:
                    gv_start = f"{gap_violation[0]//60:02d}:{gap_violation[0]%60:02d}"
                    gv_end = f"{gap_violation[-1]//60:02d}:{gap_violation[-1]%60:02d}"
                    # Длительность: разница между последней и первой минутой пропуска + 1
                    duration = gap_violation[-1] - gap_violation[0] + 1
                    #if symbol == 'CA60': print("duration ", duration)
                    low_activity.append({
                        'Ticker': symbol,
                        'Full_Interval': interval,
                        'Gap_Period': f"{gv_start}-{gv_end}",
                        'Duration_Min': duration,  # Новая колонка
                        'Schedule_Applied': sched_text,
                        'Weekday': csv_weekday,
                        'Type': 'Gap in Session'
                    })
    return pd.DataFrame(low_activity), pd.DataFrame(outside_activity)
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

# [ Функция ] фильтрации интервалов более [duration_threshold] минут <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
def filter_by_duration(df, interval_col, duration_threshold=-1):        # Оставляем только интервалы более [duration_threshold] минут
    df_gaps = df[(pd.to_timedelta(df[interval_col].str.split('-').str[1] + ':00') -  pd.to_timedelta(df[interval_col].str.split('-').str[0] + ':00')
                ).dt.total_seconds() / 60 > duration_threshold]
    return df_gaps
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>    

def add_stats_to_outside_report(df_outside, df_density):
    """
    Рассчитывает статистику активности (среднее и медиану) для каждого выявленного нарушения.
    
    Логика работы:
    1. Автоматически определяет префикс даты из матрицы плотности (поддержка мультидатовых файлов).
    2. Динамически ищет колонку с временным интервалом нарушения (Period/Interval).
    3. Для каждой строки отчета делает "срез" данных из df_density, используя строковое 
       сопоставление имен колонок (формат MM.DD.HH.MM).
    4. Рассчитывает среднее (Mean) и медиану (Median) количества тиков в указанном окне.
    5. Сортирует результат по тикерам и дням недели для удобства анализа серийных сбоев.
    
    Результат:
    Добавляет две новые колонки: 'Avg_Ticks_Outside' и 'Med_Ticks_Outside'.
    """
    if df_outside.empty:
        return df_outside

    # 1. Определяем префикс даты (например, '04.20.') из колонок матрицы
    # Это позволит коду работать с любым днем
    date_prefix = df_density.columns[0][:6] 
    
    # 2. Определяем имя колонки с интервалом (Violation_Period или Gap_Period)
    target_col = next((c for c in ['Violation_Period', 'Violation_Interval', 'Gap_Period'] if c in df_outside.columns), None)
    
    if not target_col:
        print("❌ Ошибка: Не найдена колонка с временным интервалом")
        return df_outside

    def get_row_stats(row):
        ticker = row['Ticker']
        period = row[target_col]
        
        # Базовые проверки
        if ticker not in df_density.index or not isinstance(period, str) or '-' not in period:
            return 0.0, 0.0
        
        try:
            # Превращаем "00:01-00:05" в "04.20.00.01" и "04.20.00.05"
            start_t, end_t = period.split('-')
            full_start = f"{date_prefix}{start_t.replace(':', '.')}"
            full_end = f"{date_prefix}{end_t.replace(':', '.')}"
            
            # Выбираем все колонки из df_density, которые попадают в этот диапазон
            # Это сработает даже если у вас 7200 колонок (детализация выше минуты)
            mask = (df_density.columns >= full_start) & (df_density.columns <= full_end)
            data_slice = df_density.loc[ticker, mask]
            
            if data_slice.empty:
                return 0.0, 0.0
            
            return round(data_slice.mean(), 2), round(data_slice.median(), 2)
        except:
            return 0.0, 0.0

    result_df = df_outside.copy()                                                       # Создаем копию и применяем расчет
    
    stats = result_df.apply(get_row_stats, axis=1)                                      # Применяем функцию и распределяем результат по двум новым колонкам
    result_df['Avg_Ticks_Outside'], result_df['Med_Ticks_Outside'] = zip(*stats)
    
    result_df = result_df.sort_values(by=['Ticker', 'Weekday']) # Упорядочить сначала по target_symbolу, а затем по Weekday

    return result_df

def get_symbol_violations_report_v2(df_gaps, df_extra, target_ticker):
    """
    Создает сводный отчет о нарушениях для конкретного тикера, классифицируя их по положению относительно торговой сессии.
    
    Логика работы:
    1. Извлекает данные из двух источников: df_gaps (пропуски внутри сессии) и df_extra (активность вне сессии).
    2. Унифицирует названия колонок для возможности объединения (concat).
    3. Вычисляет "позицию" нарушения (Position):
       - Для Gap in Session: определяет, произошел ли пропуск на открытии (Start), закрытии (End) или в середине сессии (Middle).
       - Для Outside Session: классифицирует данные как Pre-Open (перед открытием), Post-Close (после закрытия) или просто Outside.
    4. Парсит строку расписания Schedule_Applied, корректно обрабатывая сложные сессии с перерывами (<br>).
    5. Сортирует все инциденты хронологически по дням недели и времени начала нарушения.
    
    Результат:
    DataFrame с колонками: Full_Interval, Violation_Interval, Duration_Min, Position, Schedule_Applied, Weekday, Type.
    """
    gaps_sub = df_gaps[df_gaps['Ticker'] == target_ticker].copy()
    extra_sub = df_extra[df_extra['Ticker'] == target_ticker].copy()
    
    # Унифицируем колонки интервалов
    gaps_sub = gaps_sub.rename(columns={'Gap_Period': 'Violation_Interval'})
    extra_sub = extra_sub.rename(columns={'Violation_Period': 'Violation_Interval'})
    
    combined_df = pd.concat([gaps_sub, extra_sub], ignore_index=True)
    if combined_df.empty:
        return combined_df

    def detect_position(row):
        try:
            # Парсим время нарушения (например, 13:30-13:51)
            v_start, v_end = row['Violation_Interval'].split('-')
            
            # Парсим расписание (берем первую и последнюю сессию, если их несколько)
            # Например, "13:30-20:59" или "00:00-19:59<br>21:00-23:59"
            sched_parts = row['Schedule_Applied'].replace('<br>', '-').split('-')
            s_start = sched_parts[0]
            s_end = sched_parts[-1]

            # Логика для пропусков (Gaps)
            if row['Type'] == 'Gap in Session':
                if v_start == s_start: return "Start"
                if v_end == s_end: return "End"
                return "Middle"
            
            # Логика для лишних данных (Outside)
            else: 
                # Если лишние данные закончились ровно в момент открытия рынка
                if v_end == s_start: return "Pre-Open"
                # Если лишние данные начались ровно в момент закрытия рынка
                if v_start == s_end: return "Post-Close"
                return "Outside"
        except:
            return "Unknown"

    # Применяем определение позиции
    combined_df['Position'] = combined_df.apply(detect_position, axis=1)
    
    # Финальная сборка
    final_cols = ['Full_Interval', 'Violation_Interval', 'Duration_Min', 'Position', 'Schedule_Applied', 'Weekday', 'Type']
    
    # Сортировка по времени
    combined_df['sort_time'] = combined_df['Violation_Interval'].str.split('-').str[0]
    result = combined_df[final_cols + ['sort_time']].sort_values(['Weekday', 'sort_time'])
    
    return result.drop(columns=['sort_time'])

In [3]:
# Загружаем CSV с расписанием,  формируется в файле [ sessions_conditions_2.ipynb ]<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# '''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
file_path_symbols = Path(directory_data_temp_files)  / "df_sessions_enriched.csv" # Файл с торговыми сесcиями формируется в файле [ sessions_conditions_2.ipynb ]
print("latest_symbolTicks_df.csv:", file_path_symbols)
loader = imported["CSVLoader"](file_path_symbols, delimiter=',', encoding='utf-8', df_name='my_dataframe')  # CSVLoader для загрузки данных из файла
df_sessions = loader.load_data()
imported["pd_set_option"]("df_sessions", df_sessions, 5)                                                    # Вывод ДФ для проверки

unique_symbols = df_sessions['name_s'].unique()                                                             # Уникальные символы по колонке 'name_s'

schedule_map = {}                                                               # Пре-процессинг расписания (чтобы не парсить строки в каждом цикле)
for _, row in df_sessions.iterrows():                                           # Создаем словарь: {symbol: {weekday: set_of_minutes}}
    symbol = row['name_s']
    schedule_map[symbol] = {}
    for d in range(7): # 0-6
        col = f"1_{d}"                                                          # Берем тип 1 (trade) для проверки. Если нужно Quote, меняем на 0_
        schedule_map[symbol][d] = get_minutes_from_session(row.get(col, ""))

latest_symbolTicks_df.csv: c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\trading_conditions\working_data_files\df_sessions_enriched.csv
[class CSVLoader]: DataFrame 'my_dataframe' успешно создан из 'c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\trading_conditions\working_data_files\df_sessions_enriched.csv'.

df_sessions  (1,641 строк × 21 колонок)


,name_s,displayName_s,marketId_s,name_market,tradeMode_s,symbolId,0_0,1_0,0_1,1_1,0_2,1_2,0_3,1_3,0_4,1_4,0_5,1_5,0_6,1_6,id
0,AUDCAD,AUD / CAD,2.0,Minor,4.0,1,21:00-23:59,21:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-20:59,00:00-20:59,NaN,NaN,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1640,AU.N,Anglo Gold,33.0,CFDs - Stocks United States,4.0,10213,NaN,NaN,13:30-19:59,13:30-19:59,13:30-19:59,13:30-19:59,13:30-19:59,13:30-19:59,13:30-19:59,13:30-19:59,13:30-19:59,13:30-19:59,NaN,NaN,33.0


In [ ]:
symbol_to_check = ['BRTSPOT', 'Gas', 'Gasoil', 'NG', 'WTISPOT', 'AEX25', 'BEL20', 'CA60', 'CHINA50', 'DJC', 'DJI', 'DJT', 'DJU', 'ESTX50', 'FTSE', 'HKMYA', 'HSI', 'IBEX', 'ITALY40', 'KOSPI', 'ME0000', 'N225', 'NACOMP', 'NDX', 'NIFTY50', 'NO25', 'NYSEI', 'OMXS30', 'RUT', 'SG25', 'SLI', 'SMI', 'SP500', 'SPCOMP', 'TOPIX', 'VIX', 'VIX_SPOT', 'WIG20', 'Cocoa', 'Copper', 'CornX', 'Cotton', 'OJ', 'RghRice', 'Soybean', 'Sugar', 'SugarUK', 'Wheat', 'WheatX', 'AUS200', 'CAC40', 'CAN60', 'CH20', 'EUR50', 'GER40', 'HKD50', 'IT40', 'NL25', 'SAFRI40', 'SPAIN35', 'UK100', 'US100', 'US2000', 'US30']
#dot_big_sql_main_cred= "own_platform\\credits\\own_platforn_sql_main_stage_01.txt"

dot_big_sql_main_prices_cred= "own_platform\\credits\\own_platforn_sql_prod_prices.txt"

In [ ]:


"""{
'host':'10.0.0.8',
'user':'pma.y.d',
'password':'*************',
'database':'main'
}"""
dot_big_sql_main_cred= "own_platform\\credits\\own_platforn_sql_main_main_01.txt"
symbols_markets_df  = imported["get_sql_tab"]("SELECT * FROM `symbolsMarkets`",     dot_big_sql_main_cred)

"""{
'host':'10.0.0.8',
'user':'pma.y.d',
'password':'*************',
'database':'prices'
}"""
dot_big_sql_main_prices_cred= "own_platform\\credits\\own_platforn_sql_prod_prices.txt"
symbols_markets_df = imported["get_sql_tab"]("SELECT * FROM `candlesticksM1__26_05`", dot_big_sql_main_prices_cred)

получаем данные SELECT * FROM `symbolsMarkets`; из: 10.0.0.8 pma.y.d main
получаем данные SELECT * FROM `candlesticksM1__26_05`; из: 10.0.0.8 pma.y.d prices


OperationalError: (pymysql.err.OperationalError) (1142, "SELECT command denied to user 'pma.y.d'@'10.0.0.2' for table 'candlesticksM1__26_05'")
[SQL: SELECT * FROM `candlesticksM1__26_05`;]
(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [24]:
# Более безопасный и быстрый способ формирования списка ID
valid_ids = df_sessions[df_sessions['name_s'].isin(symbol_to_check)]['symbolId'].dropna().unique()

if len(valid_ids) > 0:
    #ids_str = ",".join(map(str, valid_ids))
    ids_str = "988, 70"
    print(f"✅ Найдено {len(valid_ids)} валидных ID для тикеров из symbol_to_check. IDs: {ids_str}")
    #query = f"SELECT * FROM prices_stage_01.candlesticksM1__26_04 WHERE symbolId IN ({ids_str})"

    #symbols_markets_df = imported["get_sql_tab"](query, dot_big_sql_main_cred)
    #symbols_markets_df = imported["get_sql_tab"](query, dot_big_sql_main_cred, sep="")
    query = f"SELECT * FROM `prices_stage_01`.`candlesticksM1__26_04` WHERE `symbolId` IN ({ids_str})"
    print(f"Сформированный SQL-запрос: {query}")
    #symbols_markets_df = imported["get_sql_tab"](query, dot_big_sql_main_cred, sep="")
    symbols_markets_df = imported["get_sql_tab"](query, dot_big_sql_main_cred)
else:
    print("❌ Тикеры не найдены в df_sessions")

✅ Найдено 64 валидных ID для тикеров из symbol_to_check. IDs: 988, 70
Сформированный SQL-запрос: SELECT * FROM `prices_stage_01`.`candlesticksM1__26_04` WHERE `symbolId` IN (988, 70)
получаем данные SELECT * FROM `prices_stage_01`.`candlesticksM1__26_04` WHERE `symbolId` IN (988, 70); из: 10.0.0.8 pma.y.d main


OperationalError: (pymysql.err.OperationalError) (1142, "SELECT command denied to user 'pma.y.d'@'10.0.0.2' for table 'candlesticksM1__26_04'")
[SQL: SELECT * FROM `prices_stage_01`.`candlesticksM1__26_04` WHERE `symbolId` IN (988, 70);]
(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [ ]:


# Нужно сопоставить именам из списка, идентификаторы, которые взять из df_sessions["symbolId"] сопоставлять по df_sessions["name_s"]
# По этим идентификаторам выполнит SQL запрос:
query = "SELECT * FROM `prices_stage_01.candlesticksM1__26_04_2024` WHERE symbolId IN ({})".format(
    ",".join(str(df_sessions[df_sessions['name_s'] == sym]['symbolId'].values[0]) for sym in symbol_to_check if sym in df_sessions['name_s'].values))
symbols_markets_df  = imported["get_sql_tab"]   (query,     dot_big_sql_main_cred)

In [ ]:

pd.set_option('display.max_colwidth', None)             # Устанавливаем бесконечную или очень большую ширину колонки
#symbol_to_check = ['XAGUSD', 'XAUUSD',"XPTUSD", "XPDUSD"]
symbol_to_check = ['BRTSPOT', 'Gas', 'Gasoil', 'NG', 'WTISPOT', 'AEX25', 'BEL20', 'CA60', 'CHINA50', 'DJC', 'DJI', 'DJT', 'DJU', 'ESTX50', 'FTSE', 'HKMYA', 'HSI', 'IBEX', 'ITALY40', 'KOSPI', 'ME0000', 'N225', 'NACOMP', 'NDX', 'NIFTY50', 'NO25', 'NYSEI', 'OMXS30', 'RUT', 'SG25', 'SLI', 'SMI', 'SP500', 'SPCOMP', 'TOPIX', 'VIX', 'VIX_SPOT', 'WIG20', 'Cocoa', 'Copper', 'CornX', 'Cotton', 'OJ', 'RghRice', 'Soybean', 'Sugar', 'SugarUK', 'Wheat', 'WheatX', 'AUS200', 'CAC40', 'CAN60', 'CH20', 'EUR50', 'GER40', 'HKD50', 'IT40', 'NL25', 'SAFRI40', 'SPAIN35', 'UK100', 'US100', 'US2000', 'US30']
#symbol_to_check = ['Lead', '1INCHUSD', 'AUDCAD', 'XAUUSD', 'XAGUSD', 'AGUAS-A.SAN', '1COV.DE', '1301.JP', 'A.N']
df = df_status[df_status['Ticker'].isin(symbol_to_check)].copy() # Если нужно сохранить оригинальный df_status, иначе можно работать с ним напрямую
imported["pd_set_option"]("df", df, 5)

In [ ]:
# Технический код для проверки <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
df_filtered = df[df["Ticker"] == 'WTISPOT']
print("symbols_sessions_df[\"symbolId\"] == 1190: ", len(df_filtered))
imported["pd_set_option"]("df_filtered", df_filtered, 50)

>>>>>>>>>>>>>>>>>>>>>Закончили подготовку таблицы котировок

Начинаем анализ котировок и торговых сесий

In [ ]:
# Создание отчёта по наличию / отсутствию котировок в разрезе расписания <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# ''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
start_date = date_start_limit
analysis_year = pd.to_datetime(start_date).year #Используем год из начальной даты для формирования объектов datetime в функции analyze_status_intervals_v2
df_gaps, df_extra = analyze_status_intervals_v2(df, schedule_map, analysis_year, df_sessions) # отчёт по наличию / отсутствию котировок в разрезе расписания

In [ ]:
# Поиск отсутствия котировок в торговых сессиях <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# ''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
duration_threshold= 4                                                                   # период, менее которого разрывы не считаются значимыми (в минутах)
df_gaps_filtered = filter_by_duration(df_gaps, "Gap_Period", duration_threshold).copy()
print(f"Найдено дыр в сессиях: {len(df_gaps_filtered)}")
#df_gaps_filtered = df_gaps_filtered.drop(columns=['Full_Interval'])
imported["pd_set_option"](f"[ df_gaps_filtered ] \n дыры в сессиях [ {start_date} ]; разрывы более [ {duration_threshold} ] минут \n", df_gaps_filtered, 400)
"""Full_Interval - полный интервал отсутствия котировок, который был найден в df_status"""

In [ ]:
# котировки вне сессий
df_extra_filtered = filter_by_duration(df_extra, "Violation_Period", duration_threshold= 4).copy()
print(f"Найдено котировок вне сессий: {len(df_extra_filtered)}")
#imported["pd_set_option"]("df_extra", df_extra, 100)
imported["pd_set_option"]("[ df_extra_filtered ] котировки вне сессий", df_extra_filtered, 50)

In [ ]:
# Рассчитывает статистику активности (среднее и медиану) для каждого выявленного нарушения <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# '''''''''''''''''''''
df_outside_with_stats = add_stats_to_outside_report(df_extra, df_density)
duration_threshold = 1 # отфильтровать по длительности интервала, если нужно (например, оставить только те, где Duration_Min > 15)
df_outside_with_stats = df_outside_with_stats[df_outside_with_stats['Duration_Min'] > duration_threshold].copy()
imported["pd_set_option"]("[ df_outside_with_stats ] Результаты внесессионной активности", df_outside_with_stats, 400)   # Вывод результата

In [ ]:
# Технический код для проверки <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
df_filtered = df_outside_with_stats[df_outside_with_stats["Ticker"] == 'WTISPOT']
imported["pd_set_option"]("df_outside_with_stats", df_filtered, 50)

In [ ]:
# Создает сводный отчет о нарушениях для конкретного тикера, классифицируя их по положению относительно торговой сессии.
Duration_Min = -1
target_symbol = 'WTISPOT'
df_report = get_symbol_violations_report_v2(df_gaps, df_extra, target_symbol)
# Фильтруем по длительности Duration_Min
df_report = df_report[df_report['Duration_Min'] > Duration_Min]
#imported["pd_set_option"](f"{target_symbol}; Отчет по gaps: {target_symbol}", df_report, 50)

In [ ]:
# Технический код для проверки <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
df_filtered = df_sessions[df_sessions["name_s"] == 'WTISPOT']
print("symbols_sessions_df[\"symbolId\"] == 1190: ", len(df_filtered))
imported["pd_set_option"]("df_filtered", df_filtered, 50)

In [ ]:
import matplotlib.pyplot as plt

# 1. Настройки фильтрации
target_ticker = 'Gas'          # Тикер для анализа
time_range = "21:00-23:59"      # Промежуток ЧЧ:ММ - ЧЧ:ММ
date_range = "04.19.-04.24."    # Промежуток ДАТ ММ.ДД. - ММ.ДД.

# Парсим настройки
t_start, t_end = time_range.replace(':', '.'), time_range.split('-')[1].replace(':', '.')
d_start, d_end = date_range.split('-')[0], date_range.split('-')[1]

# 2. Получаем уникальные дни из колонок, попадающие в диапазон дат
all_cols = df_density.columns
target_dates = sorted(list(set([c[:6] for c in all_cols if d_start <= c[:6] <= d_end])))

# 3. Построение графиков
fig, axes = plt.subplots(len(target_dates), 1, figsize=(15, 2 * len(target_dates)), sharey=True)
if len(target_dates) == 1: axes = [axes] # Если день один, делаем список для итерации

for i, day_prefix in enumerate(target_dates):
    # Фильтруем колонки для конкретного дня и диапазона времени
    day_cols = [c for c in all_cols if c.startswith(day_prefix) and t_start <= c[6:] <= t_end]
    
    if not day_cols:
        axes[i].set_title(f"No data for {day_prefix}")
        continue
        
    data_to_plot = df_density.loc[target_ticker, day_cols]
    
    # Отрисовка
    axes[i].bar(range(len(data_to_plot)), data_to_plot.values, color='skyblue', edgecolor='navy')
    
    # Настройки осей для каждого дня
    axes[i].set_title(f"Tick Density: {target_ticker} | Date: {day_prefix.strip('.')}", fontsize=12)
    axes[i].set_ylabel("Ticks Count")
    
    # Подписи времени (X) - берем только ЧЧ.ММ из полного названия колонки
    step = max(1, len(day_cols) // 40) # Чтобы не частить с подписями
    axes[i].set_xticks(range(0, len(day_cols), step))
    axes[i].set_xticklabels([c[6:] for c in day_cols[::step]], rotation=45)
    axes[i].grid(axis='y', linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# 1. Настройки
target_ticker = 'Wheat'  # Пример тикера с двумя сессиями
date_range = "04.19.-04.24."

# Получаем строку расписания из вашего DF (df_outside_with_stats или df_total_report_enriched)
# Берем первую попавшуюся запись для этого тикера
try:
    raw_sched = df_outside_with_stats[df_outside_with_stats['Ticker'] == target_ticker]['Schedule_Applied'].iloc[0]
except:
    raw_sched = "00:00-23:59" # Дефолт, если не нашли

# Парсим сессии (разделяем по <br> и затем по дефису)
sessions = []
for part in raw_sched.split('<br>'):
    if '-' in part:
        start_t, end_t = part.split('-')
        sessions.append((start_t.strip().replace(':', '.'), end_t.strip().replace(':', '.')))

# 2. Подготовка дат и осей
all_cols = df_density.columns
target_dates = sorted(list(set([c[:6] for c in all_cols if d_start <= c[:6] <= d_end])))

fig, axes = plt.subplots(len(target_dates), 1, figsize=(15, 2 * len(target_dates)), sharey=True)
if len(target_dates) == 1: axes = [axes]

for i, day_prefix in enumerate(target_dates):
    day_cols = [c for c in all_cols if c.startswith(day_prefix)]
    if not day_cols: continue
    
    data_to_plot = df_density.loc[target_ticker, day_cols]
    times_only = [c[6:] for c in day_cols]
    
    # Отрисовка данных
    axes[i].bar(range(len(data_to_plot)), data_to_plot.values, color='skyblue', log=True, zorder=3)
    
    # 3. ЛОГИКА ЗАЛИВКИ (Inverse Session Mask)
    # Создаем маску: по умолчанию все время "вне сессии" (True)
    outside_mask = [True] * len(times_only)
    
    for s_start, s_end in sessions:
        for idx, t in enumerate(times_only):
            if s_start <= t <= s_end:
                outside_mask[idx] = False # Это время внутри одной из сессий
        
        # Рисуем границы сессий
        if s_start in times_only:
            axes[i].axvline(times_only.index(s_start), color='red', linestyle='--', alpha=0.6, zorder=4)
        if s_end in times_only:
            axes[i].axvline(times_only.index(s_end), color='red', linestyle='--', alpha=0.6, zorder=4)

    # Закрашиваем только те участки, где outside_mask == True
    # Для этого ищем непрерывные интервалы True
    start_idx = None
    for idx, is_outside in enumerate(outside_mask):
        if is_outside and start_idx is None:
            start_idx = idx
        elif not is_outside and start_idx is not None:
            axes[i].axvspan(start_idx, idx - 1, color='gray', alpha=0.2, zorder=1)
            start_idx = None
    if start_idx is not None: # Докрашиваем до конца дня, если нужно
        axes[i].axvspan(start_idx, len(outside_mask) - 1, color='gray', alpha=0.2, zorder=1)

    # Оформление
    axes[i].set_title(f"Ticker: {target_ticker} | Date: {day_prefix} | Schedule: {raw_sched.replace('<br>', ' & ')}")
    step = max(1, len(day_cols) // 24)
    axes[i].set_xticks(range(0, len(day_cols), step))
    axes[i].set_xticklabels(times_only[::step], rotation=45)
    axes[i].grid(True, which="both", axis='y', linestyle='--', alpha=0.3, zorder=0)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. Настройки фильтрации
target_ticker = 'CHINA50'          # Тикер для анализа
time_range = "00:00-23:59"      # Промежуток ЧЧ:ММ - ЧЧ:ММ
date_range = "04.19.-04.24."    # Промежуток ДАТ ММ.ДД. - ММ.ДД.

# Парсим настройки
t_start = time_range.split('-')[0].replace(':', '.')
t_end = time_range.split('-')[1].replace(':', '.')
d_start, d_end = date_range.split('-')[0], date_range.split('-')[1]

# 2. Получаем уникальные дни
all_cols = df_density.columns
target_dates = sorted(list(set([c[:6] for c in all_cols if d_start <= c[:6] <= d_end])))

# 3. Построение графиков
fig, axes = plt.subplots(len(target_dates), 1, figsize=(15, 2 * len(target_dates)), sharey=True)
if len(target_dates) == 1: axes = [axes]

for i, day_prefix in enumerate(target_dates):
    day_cols = [c for c in all_cols if c.startswith(day_prefix) and t_start <= c[6:] <= t_end]
    
    if not day_cols:
        axes[i].set_title(f"No data for {day_prefix}")
        continue
        
    data_to_plot = df_density.loc[target_ticker, day_cols]
    
    # Отрисовка. Используем логарифмическую шкалу для оси Y
    axes[i].bar(range(len(data_to_plot)), data_to_plot.values, color='skyblue', edgecolor='navy', log=True)
    
    # Альтернативный способ задания логарифмической шкалы:
    # axes[i].set_yscale('log') 

    axes[i].set_title(f"Log Scale Tick Density: {target_ticker} | Date: {day_prefix.strip('.')}", fontsize=12)
    axes[i].set_ylabel("Ticks Count (Log Scale)")
    
    # Подписи времени
    step = max(1, len(day_cols) // 20)
    axes[i].set_xticks(range(0, len(day_cols), step))
    axes[i].set_xticklabels([c[6:] for c in day_cols[::step]], rotation=45)
    
    # Добавляем сетку для логарифмических делений
    axes[i].grid(True, which="both", axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:


target_symbol = 'Gasoil'
df_report = get_symbol_violations_report_v2(df_gaps, df_extra, target_symbol)
imported["pd_set_option"](f"Отчет по нарушениям: {target_symbol}", df_report, 50)

In [ ]:
def analyze_anomaly_typicality(df_all_violations):
    if df_all_violations.empty:
        return pd.DataFrame()

    df = df_all_violations.copy()
    
    # --- ИСПРАВЛЕНИЕ ОШИБКИ: Унификация названия колонки ---
    # Проверяем, какое имя колонки используется в DF
    if 'Gap_Period' in df.columns:
        col_name = 'Gap_Period'
    elif 'Violation_Interval' in df.columns:
        col_name = 'Violation_Interval'
    elif 'Violation_Period' in df.columns:
        col_name = 'Violation_Period'
    else:
        # Если вдруг колонки нет, выведем список доступных для отладки
        raise KeyError(f"Не найдена колонка с интервалом. Доступные колонки: {list(df.columns)}")

    # 1. Извлекаем время начала нарушения
    df['Start_Time'] = df[col_name].astype(str).str.split('-').str[0]
    
    # Проверка на наличие колонки Position (если ее еще нет)
    if 'Position' not in df.columns:
        df['Position'] = 'N/A'

    # 2. Группируем по времени, типу и позиции
    # Используем только те колонки, которые гарантированно есть
    group_cols = ['Start_Time', 'Type', 'Position']
    
    patterns = df.groupby(group_cols).agg({
        'Ticker': ['count', lambda x: ', '.join(x.unique()[:5]) + ('...' if len(x.unique()) > 5 else '')],
        'Duration_Min': 'mean'
    }).reset_index()
    
    # Плоские имена колонок после агрегации
    patterns.columns = ['Start_Time', 'Type', 'Position', 'Occurrences', 'Example_Tickers', 'Avg_Duration']
    
    # 3. Сортировка и классификация
    patterns = patterns.sort_values(by='Occurrences', ascending=False)
    
    # Считаем типичными те, что повторились более 5 раз (можно настроить)
    threshold = 5 
    patterns['Typicality'] = patterns['Occurrences'].apply(
        lambda x: 'Systemic / Typical' if x >= threshold else 'Random / Unique'
    )
    return patterns

df_total_report = pd.concat([df_gaps, df_extra], ignore_index=True)

In [ ]:
# Применяем к объединенному датафрейму по всем символам
df_all_patterns = analyze_anomaly_typicality(df_total_report)

In [ ]:
imported["pd_set_option"]("Анализ типичности нарушений", df_all_patterns, 50)

In [ ]:
def get_symbol_violations_report(df_gaps, df_extra, target_ticker):
    """
    Собирает все типы нарушений по конкретному тикеру в одну таблицу.
    """
    # 1. Фильтруем оба датафрейма по нужному тикеру
    gaps_sub = df_gaps[df_gaps['Ticker'] == target_ticker]
    extra_sub = df_extra[df_extra['Ticker'] == target_ticker]
    
    # 2. Приводим колонки к единому названию для объединения
    # В df_gaps колонка называется 'Gap_Period', в df_extra — 'Violation_Period'
    # Переименуем их в общее 'Activity_Period'
    gaps_sub = gaps_sub.rename(columns={'Gap_Period': 'Violation_Interval'})
    extra_sub = extra_sub.rename(columns={'Violation_Period': 'Violation_Interval'})
    
    # 3. Объединяем таблицы
    combined_df = pd.concat([gaps_sub, extra_sub], ignore_index=True)
    
    # Если данных нет, возвращаем пустой DF с нужными колонками
    if combined_df.empty:
        return pd.DataFrame(columns=['Violation_Interval', 'Duration_Min', 'Schedule_Applied', 'Weekday', 'Type'])

    # 4. Выбираем только нужные колонки
    final_cols = ['Violation_Interval', 'Duration_Min', 'Schedule_Applied', 'Weekday', 'Type']
    combined_df = combined_df[final_cols]
    
    # 5. Хронологическая сортировка
    # Сначала по дню недели, затем по времени начала нарушения
    combined_df['sort_time'] = combined_df['Violation_Interval'].str.split('-').str[0]
    combined_df = combined_df.sort_values(by=['Weekday', 'sort_time']).drop(columns=['sort_time'])
    
    return combined_df

# --- Пример использования ---
target_symbol = 'TOPIX'
df_report = get_symbol_violations_report(df_gaps, df_extra, target_symbol)
imported["pd_set_option"](f"Отчет по нарушениям: {target_symbol}", df_report, 50)

# Вывод в Jupyter или сохранение в HTML
# df_to_html(df_report, f"Отчет по нарушениям: {target_symbol}", f"report_{target_symbol}")